# Tools
## 工具调用的方式
工具就是明确定义了输入和输出的**可调用函数**。工具调用 (Tool Calling) 也被称为 `函数调用(Function Calling)`。

### 直接调用

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
    city: 城市名称，如"北京"、"上海"
    返回:
    天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

# 使用 .invoke() 方法
result = get_weather.invoke({"city": "北京"})
print(result) # 输出：北京晴天，温度 15°C

北京晴天，温度 15°C


### 基于模型进行调用

In [3]:
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen
from rich import print as rprint

# 将env文件中的变量加载为环境变量
#override=True：表示.env优先
load_dotenv(override=True)

model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

In [6]:
from langchain_core.tools import tool
# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度 15°C"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])
# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
rprint(response)
# 检查 AI 是否要调用工具
if response.tool_calls:
    rprint("AI 想调用工具：", response.tool_calls)
else:
    rprint("AI 直接回答：", response.content)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'用户想知道北京的天气。我可以使用get_weather工具来获取北京当前的天气信息。让我调用这个工具。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 69,
            'prompt_tokens': 276,
            'total_tokens': 345,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 23,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'dashscope',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': None,
        'id': 'chatcmpl-6fadf395-e608-9914-ad54-f9489b525671',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f0ede-8430-7bf1-8a40-4b1783ab0a1f-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_d9c36ed2d796489cbbeab4f5',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 276,
        'output_tokens': 69,
        'total_tokens': 345,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 23}
    }
)

AI 想调用工具：
[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_d9c36ed2d796489cbbeab4f5', 'type': 'tool_call'}]

### 从Message流转看工具的调用

In [ ]:
# 使用tools
from langchain.messages import HumanMessage, ToolMessage
@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]
# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息，这块主动调用工具
        # 添加到消息列表中
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)
rprint("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
rprint("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
rprint(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>


=====================> messages <=====================

================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_ad6403a6f459496f8c59af4d)
 Call ID: call_ad6403a6f459496f8c59af4d
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~


=====================> messages <=====================

final_response: 
content='今天北京的天气是 **晴朗** ☀️，是个好天气！' additional_kwargs={'refusal': None, 'reasoning_content': 'The 
weather in Beijing is sunny/clear today.'} response_metadata={'token_usage': {'completion_tokens': 27, 
'prompt_tokens': 336, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': None, 
'audio_tokens': None, 'reasoning_tokens': 10, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': 
{'audio_tokens': None, 'cached_tokens': 256}}, 'model_provider': 'dashscope', 'model_name': 'deepseek-v4-flash', 
'system_fingerprint': None, 'id': 'chatcmpl-1480de7c-da62-95ec-b369-0037de2d0a7d', 'finish_reason': 'stop', 
'logprobs': None} id='lc_run--019f0eea-7481-7592-8143-02f50450109a-0' tool_calls=[] invalid_tool_calls=[] 
usage_metadata={'input_tokens': 336, 'output_tokens': 27, 'total_tokens': 363, 'input_token_details': 
{'cache_read': 256}, 'output_token_details': {'reasoning': 10}}

被 `@tool` 修饰的函数可以调用 `invoke` 接收模型返回的入参信息执行函数，并返回`ToolMessage 实例`，我们不再需要手动拼接 `ToolMessage` 。

In [ ]:
# 不使用tools
from langchain.messages import HumanMessage, ToolMessage
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗"

# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]
# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
    # 拼接出ToolMessage实例
        tool_response = ToolMessage(content=get_weather(**tool_call["args"]),tool_call_id=tool_call["id"],name=tool_call["name"])
        messages.append(tool_response)

rprint("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
rprint("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
rprint(f"final_response: \n{final_response}")

=====================> messages <=====================

================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_8330412c32284964aeda0405)
 Call ID: call_8330412c32284964aeda0405
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗


=====================> messages <=====================

final_response: 
content='今天北京的天气是 **晴朗** 
☀️，天气不错，适合出门活动！不过具体温度信息没有提供，建议您出门前查看一下实时温度，适当增减衣物。' 
additional_kwargs={'refusal': None, 'reasoning_content': 'The tool returned "北京天气晴朗" which means "Beijing 
weather is sunny/clear".'} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 335, 
'total_tokens': 393, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 
'reasoning_tokens': 18, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 
'cached_tokens': 0}}, 'model_provider': 'dashscope', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': None,
'id': 'chatcmpl-244dc02f-72a1-9100-b4d3-0a22cd342282', 'finish_reason': 'stop', 'logprobs': None} 
id='lc_run--019f0ee5-4ad4-76a1-a052-c6facd974d21-0' tool_calls=[] invalid_tool_calls=[] 
usage_metadata={'input_tokens': 335, 'output_tokens': 58, 'total_tokens': 393, 'input_token_details': 
{'cache_read': 0}, 'output_token_details': {'reasoning': 18}}

总体流程：
1. 初始化模型和工具
2. 创建HumanMessage
3. 调用模型（Invoke）
4. 生成AIMessage(包含tool_calls)
5. 保存AI响应到消息列表
6. 调用工具并生成ToolMessage
7. 将ToolMessage添加到消息列表
8. 调用模型（Invoke）
9.  返回最终的AIMessage

## 工具的定义方式
### 不使用@tool 
#### 模型绑定工具并发送请求


In [14]:
from rich import print as rprint
# 定义工具
def get_weather(city: str):
    return f"{city}天气晴朗"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke(
    "今天北京天气如何"
)
# rprint(response)
rprint(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_2634b92ae0944abdb53d15d4', 'type': 'tool_call'}]

### 工具描述的各个部分详解
#### convert_to_openai_tool
执行 `model.bind_tools([get_weather])` ，底层最终会调用 `convert_to_openai_tool` 生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [17]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich.console import Console
from rich.theme import Theme
from rich.json import JSON

def get_weather(city: str):
    return f"{city}天气晴朗"

# 深色主题下自定义 JSON 配色（默认 key 是深蓝色，很难看清）
JSON_THEME = Theme({
    "json.key": "bold bright_yellow",   # 键名
    "json.str": "bright_green",         # 字符串
    "json.number": "bright_cyan",       # 数字
    "json.bool": "bright_magenta",      # 布尔
    "json.null": "dim white",           # null
    "json.brace": "white",
    "json.bracket": "white",
    "json.colon": "white",
    "json.comma": "white",
})

Console(theme=JSON_THEME).print(JSON.from_data(convert_to_openai_tool(get_weather)))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "",
    "parameters": {
      "properties": {
        "city": {
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    }
  }
}